# Calibración de la Cámara usando OpenCV

## 0. Resumen
1. Patrón
2. Sacar fotos desde diferentes puntos de vista
3. Encontrar esquinas
4. Ecuaciones de proyección patrón->foto

\begin{equation}
s \begin{bmatrix}
u_i \\ v_i \\ 1
\end{bmatrix} =
\begin{bmatrix}
K
\end{bmatrix}
\begin{bmatrix}
R_k | t_k
\end{bmatrix}
\begin{bmatrix}
X_i \\ Y_i \\ Z_i \\ 1
\end{bmatrix}
\end{equation}

5. Hallar K, R_k, t_k y de yapa los coeficientes de distorsión.
6. Rectificar la imagen  
7. Bonus: dibujar en 3D  

In [ ]:
import cv2
import numpy as np
import glob
import supervision as sv

## Fotos desde distintos puntos de vista

In [ ]:
# Carpeta con las fotos:
calib_fnames = glob.glob('./calibration_images/*')

# visualizamos una imagen de calibración:
img = cv2.imread(calib_fnames[0])
sv.plot_image(img)

In [ ]:
PATTERN_SIZE = np.array((11, 8)) - 1

In [ ]:
# Prueba rapida para la 1er imagen
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
ret, corners = cv2.findChessboardCorners(gray, PATTERN_SIZE, None)
if ret: 
    cv2.drawChessboardCorners(img, PATTERN_SIZE, corners, ret)
else:
    print("No se encontraron esquinas en la imagen de calibración.")
sv.plot_image(img)

## 3, 4, 5. Identificación de Esquinas, Encontrar Matriz de Cámara

In [ ]:
# Lista de los puntos que vamos a reconocer en el mundo
# objp={(0,0,0), (1,0,0), (2,0,0) .... }
# corresponden a las coordenadas en el tablero de ajedrez.
objp = np.zeros((np.prod(PATTERN_SIZE), 3),  dtype=np.float32)
objp[:, :2] = np.mgrid[0:PATTERN_SIZE[0], 0:PATTERN_SIZE[1]].T.reshape(-1, 2)
objp

In [ ]:
# Lista de todos los puntos que vamos a recolectar
obj_points = list() # grilla predefinida en el mundo
img_points = list() # puntos en el plano de la imagen

for image_fname in calib_fnames:
    img = cv2.imread(image_fname)
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) 
    ret, corners = cv2.findChessboardCorners(img_gray, PATTERN_SIZE, None)
    if ret:
        obj_points.append(objp)
        img_points.append(corners)

## 5. Calibración
Listo con la identificación de puntos, ahora a calibrar


In [ ]:
h, w = img.shape[:2]
ret, K, dist, rvecs, tvecs = cv2.calibrateCamera(
    obj_points, img_points, (w, h), cameraMatrix=None, distCoeffs=None)

print(tvecs[0])

print(f'Camera Matrix:\n{K}')
print(f'Dist. Coeff.: \n{dist}')
print(h, w)

In [ ]:
img_0 = cv2.imread(calib_fnames[0])
sv.plot_image(img_0)
undistorted = cv2.undistort(img, K, dist)
sv.plot_images_grid(images=[img, undistorted],
                    titles=['Original', 'Undistorted'],
                    grid_size=(1, 2))

## Dibujando en el espacio

In [ ]:
# Funcion para dibujar los ejes del mundo 3D en la imagen
def draw(img, corners, imgpts):
    corner = tuple(corners[0].ravel().astype("int32"))
    imgpts = imgpts.astype("int32")
    img = cv2.line(img, corner, tuple(imgpts[0].ravel()), (255,0,0), 5)
    img = cv2.line(img, corner, tuple(imgpts[1].ravel()), (0,255,0), 5)
    img = cv2.line(img, corner, tuple(imgpts[2].ravel()), (0,0,255), 5)
    return img

In [ ]:
img = cv2.imread(calib_fnames[6])
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
undistorted = cv2.undistort(img, K, dist)
ret, corners = cv2.findChessboardCorners(gray, PATTERN_SIZE, None)

axis = np.float32([[3,0,0], [0,3,0], [0,0,-3]]).reshape(-1,3)

ret, rvecs, tvecs = cv2.solvePnP(objp, corners, K, dist)
proj_pts, jac = cv2.projectPoints(axis, rvecs, tvecs, K, dist)
img2show = draw(img, corners, proj_pts)
sv.plot_image(img2show)


In [ ]:
tvecs